# Autism terms: psychology journals vs Google Books
Run the cells top to bottom; each step shows its own result. Change the unit in step 2 and re-run from there.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, grangercausalitytests
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 4)

## 1. Load

In [ ]:
df = pd.read_csv("autism_timeseries.csv", keep_default_na=True)
for c in ("era", "variant"):
    df[c] = df[c].fillna("")
groups = df.loc[df["level"] == "group", "group"].unique().tolist()
groups

## 2. Choose a unit and the settings

In [ ]:
GROUP = "autism/autistic"   # one of the groups above
MIN_TOKENS = 100_000        # a year needs this many corpus words to be used
SMOOTH = 5                  # rolling-mean window in years (1 = none)
MAX_LAG = 10                # years
FIRST_PPM = 0.1             # per-million level for "first year above"

In [ ]:
sub = df[(df["level"] == "group") & (df["group"] == GROUP)].set_index("year")
sub[["corpus_count", "corpus_tokens", "corpus_per_million", "google_per_million"]].loc[1930:2019].iloc[::10]

## 3. Put both series on the same years
log(1 + per million), years with too little corpus text dropped (and interpolated), then smoothed.

In [ ]:
d = sub[sub["corpus_tokens"].fillna(0) >= MIN_TOKENS]
d = d.loc[d.index.min(): d.index.max()]
raw = d[["corpus_per_million", "google_per_million"]].reindex(range(d.index.min(), d.index.max() + 1))
s = np.log1p(raw.rename(columns={"corpus_per_million": "corpus", "google_per_million": "google"}))
s = s.interpolate(limit_area="inside")
if SMOOTH > 1:
    s = s.rolling(SMOOTH, center=True, min_periods=1).mean()
print(f"{len(s)} common years: {s.index.min()}-{s.index.max()}")
s.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
s.plot(ax=ax[0], title=GROUP + " (log(1 + per million))")
scaled = (s - s.min()) / (s.max() - s.min())
scaled.plot(ax=ax[1], title="each scaled to its own range")
plt.tight_layout()

## 4. Adoption: trend and timing

In [ ]:
def level_year(x, frac):
    lo, hi = x.min(), x.max()
    if hi <= lo:
        return np.nan
    hit = x.index[(x - lo) / (hi - lo) >= frac]
    return hit[0] if len(hit) else np.nan

rows = []
for which, col in (("corpus", "corpus_per_million"), ("google", "google_per_million")):
    x = s[which]
    tau, p = stats.kendalltau(x.index, x.values)
    above = raw.index[raw[col] >= FIRST_PPM]
    rows.append({"corpus": which, "peak_year": x.idxmax(),
                 "year_10pct": level_year(x, .10), "year_50pct": level_year(x, .50), "year_90pct": level_year(x, .90),
                 "first_year_above_ppm": above[0] if len(above) else np.nan,
                 "kendall_tau": tau, "kendall_p": p,
                 "slope_per_decade": np.polyfit(x.index, x.values, 1)[0] * 10})
adoption = pd.DataFrame(rows).set_index("corpus")
adoption

## 5. Stationarity
ADF: H0 = unit root. KPSS: H0 = stationary. First differences are what the next two steps use.

In [ ]:
rows = []
for which in ("corpus", "google"):
    for form, x in (("level", s[which]), ("diff", s[which].diff().dropna())):
        r = {"series": which, "form": form, "n": len(x)}
        if x.nunique() > 1 and len(x) > 12:
            r["adf_p"] = adfuller(x, autolag="AIC")[1]
            r["kpss_p"] = kpss(x, regression="c", nlags="auto")[1]
        rows.append(r)
pd.DataFrame(rows)

## 6. Lead-lag: cross-correlation of first differences
Positive lag = Google Books leads the journals.

In [ ]:
a, b = s["corpus"].diff().dropna(), s["google"].diff().dropna()
n = len(a)
cc = []
for lag in range(-MAX_LAG, MAX_LAG + 1):
    x, y = (b.values[: n - lag], a.values[lag:]) if lag >= 0 else (b.values[-lag:], a.values[: n + lag])
    cc.append((lag, np.corrcoef(x, y)[0, 1]))
cc = pd.DataFrame(cc, columns=["lag", "r"]).set_index("lag")
cc["r"].plot.bar(title="corr(Google(t - lag), corpus(t)), first differences")
print("best lag:", cc["r"].abs().idxmax(), " r =", round(cc.loc[cc["r"].abs().idxmax(), "r"], 3))
print("50% adoption year, corpus minus Google:", level_year(s["corpus"], .5) - level_year(s["google"], .5))

## 7. Granger causality (first differences)
p-values per lag; small = the other series helps predict this one.

In [ ]:
x = s[["corpus", "google"]].diff().dropna()
out = {}
for cause, effect in (("google", "corpus"), ("corpus", "google")):
    try:
        res = grangercausalitytests(x[[effect, cause]], maxlag=MAX_LAG, verbose=False)
    except TypeError:
        res = grangercausalitytests(x[[effect, cause]], maxlag=MAX_LAG)
    out[f"{cause} -> {effect}"] = {k: v[0]["ssr_ftest"][1] for k, v in res.items()}
granger = pd.DataFrame(out)
granger.index.name = "lag"
granger.round(4)

## 8. Same steps for every group (optional)
Wraps steps 3, 4 and 6 in a loop and returns one table; nothing above is changed.

In [ ]:
def summarize(g):
    sb = df[(df["level"] == "group") & (df["group"] == g)].set_index("year")
    dd = sb[sb["corpus_tokens"].fillna(0) >= MIN_TOKENS]
    dd = dd.loc[dd.index.min(): dd.index.max()]
    ss = np.log1p(dd[["corpus_per_million", "google_per_million"]].reindex(range(dd.index.min(), dd.index.max() + 1))
                  .set_axis(["corpus", "google"], axis=1)).interpolate(limit_area="inside")
    if SMOOTH > 1:
        ss = ss.rolling(SMOOTH, center=True, min_periods=1).mean()
    if ss["corpus"].max() == 0 or ss["google"].max() == 0 or len(ss) < 25:
        return None
    aa, bb = ss["corpus"].diff().dropna(), ss["google"].diff().dropna()
    nn = len(aa)
    r = {}
    for lag in range(-MAX_LAG, MAX_LAG + 1):
        xx, yy = (bb.values[: nn - lag], aa.values[lag:]) if lag >= 0 else (bb.values[-lag:], aa.values[: nn + lag])
        r[lag] = np.corrcoef(xx, yy)[0, 1]
    r = pd.Series(r)
    return {"group": g, "years": f"{ss.index.min()}-{ss.index.max()}",
            "year50_corpus": level_year(ss["corpus"], .5), "year50_google": level_year(ss["google"], .5),
            "best_lag": r.abs().idxmax(), "best_r": r.loc[r.abs().idxmax()], "r_lag0": r[0]}

summary = pd.DataFrame([x for x in map(summarize, groups) if x]).set_index("group")
summary

In [ ]:
# summary.to_csv("ts_summary.csv")